In [ ]:
from importlib import reload
import test_data_model as tdm
reload(tdm)
import data_schema as ds
reload(ds)
import numpy as np
import pandas as pd
import itertools
from harbor.analysis.cross_docking import Settings

In [ ]:
def refs():
    """Sample reference structures fixture."""
    return [f"PDB{i}" for i in range(1, 10)]
refs = refs()

def ligs():
    """Sample ligands fixture."""
    return ["LIG_A", "LIG_B", "LIG_C"]
ligs = ligs()

def ref_dataframe(refs):
    """Sample reference data fixture."""
    return pd.DataFrame(
        {
            "Reference_Structure": refs,
            "Ref_Data_1": [np.random.random() for ref in refs],
            "Date": [datetime.now() - timedelta(days=i) for i in range(len(refs))],
        }
    )
ref_dataframe = ref_dataframe(refs)

def lig_dataframe(ligs):
    """Sample ligand data fixture."""
    return pd.DataFrame(
        {
            "Query_Ligand": ligs,
            "Lig_Data_1": [np.random.random() for lig in ligs],
        }
    )
lig_dataframe = lig_dataframe(ligs)

def pose_dataframe(refs, ligs):
    """Sample pose results fixture."""
    return pd.DataFrame.from_records(
        [
            {
                "Reference_Structure": ref,
                "Query_Ligand": lig,
                "RMSD": np.random.random() * 8,
                "Pose_ID": pose,
            }
            for ref, lig, pose in itertools.product(refs, ligs, range(0, 2))
        ]
    )
pose_dataframe = pose_dataframe(refs, ligs)

def ecfp_dataframe(refs, ligs):
    """Sample ECFP data fixture."""
    return pd.DataFrame.from_records(
        [
            {
                "Reference_Structure": ref,
                "Query_Ligand": lig,
                "Tanimoto": np.random.random(),
                "radius": radius,
                "bitsize": bitsize,
            }
            for ref, lig, radius, bitsize in itertools.product(
                refs, ligs, [2, 5], [2048]
            )
        ]
    )
ecfp_dataframe = ecfp_dataframe(refs, ligs)


def tanimotocombo_data(refs, ligs):
    """Sample TanimotoCombo data fixture."""
    return pd.DataFrame.from_records(
        [
            {
                "Reference_Structure": ref,
                "Query_Ligand": lig,
                "Tanimoto": np.random.random(),
                "Aligned": aligned,
            }
            for ref, lig, aligned in itertools.product(refs, ligs, ["True", "False"])
        ]
    )
tanimotocombo_data = tanimotocombo_data(refs, ligs)

In [ ]:
docking_data = ds.DockingDataModel(
        pose_data=ds.PoseData(dataframe=pose_dataframe),
        reference_data=ds.ReferenceData(dataframe=ref_dataframe,
                                        other_columns=[ds.ValueColumn(name="Ref_Data_1"),
                                                       ds.ValueColumn(name="Date")]),
        query_data=ds.QueryData(dataframe=lig_dataframe, other_columns=[ds.ValueColumn(name="Lig_Data_1")]),
        chemical_similarity_data=[
            ds.ChemicalSimilarityData(
                dataframe=ecfp_dataframe,
                name=ds.InfoColumn(name="ECFP"),
                other_columns=[
                    ds.ParamColumn(name="radius"),
                    ds.ParamColumn(name="bitsize"),
                ],
            ),
            ds.ChemicalSimilarityData(
                dataframe=tanimotocombo_data,
                name=ds.InfoColumn(name="TanimotoCombo"),
                other_columns=[ds.ParamColumn(name="Aligned")],
            ),
        ],
    )

In [ ]:
docking_data.reference_data

In [ ]:
new_model = docking_data.copy()

In [ ]:
new_model = new_model.copy()

# test how to use new model to analyze data

In [ ]:
refdf = docking_data.reference_data.dataframe

In [ ]:
refdata = docking_data.reference_data

In [ ]:
refdata.get_columns('value')

In [ ]:
refdf = refdf[refdf["Reference_Structure"] == "PDB123"]

In [ ]:
docking_data.reference_data.dataframe = refdf

In [ ]:
docking_data.reference_data

In [ ]:
docking_data.pose_data.get_columns()

In [ ]:
posedf = docking_data.pose_data.dataframe
docking_data.pose_data.dataframe = posedf[posedf["Pose_ID"] == 0]

In [ ]:
similarity_data = docking_data.get_combined_similarity_data()

In [ ]:
docking_data.get_data_as_dict()['ECFP']

In [ ]:
docking_data.pose_data

In [ ]:
docking_data.get_combined_dataframe()

# test with multindex

In [ ]:
docking_data.reference_data

In [ ]:
refdf = docking_data.reference_data.dataframe

In [ ]:
refdf = refdf.set_index(["Reference_Structure"])

In [ ]:
pose_data = docking_data.pose_data.dataframe

In [ ]:
pose_data = pose_data.set_index(["Query_Ligand", "Reference_Structure", "Pose_ID"])

# try bootstrap parallelization

In [ ]:
bootstraps = 100

In [ ]:
import logging
import warnings

from pydantic import BaseModel, Field, field_validator, model_validator
from enum import Enum, auto
from typing_extensions import Self
import abc
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from typing import Optional
from datetime import datetime, timedelta
import json
import yaml


class ModelBase(BaseModel):
    type_: str = Field(..., description="Type of model")

    @abc.abstractmethod
    def plot_name(self) -> str:
        pass

    @abc.abstractmethod
    def get_records(self) -> dict:
        pass


class EmptyModel(ModelBase):
    type_: str = Field("EmptyModel", description="Empty model")

    def plot_name(self) -> str:
        return ""

    def get_records(self) -> dict:
        return {}


class EmptyDataframeModel(EmptyModel):
    """
    A model that does nothing to the dataframe.
    """

    type_: str = Field("EmptyDataframeModel", description="Empty dataframe model")

    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        return df


class SettingsBase(BaseModel):
    def get_descriptions(self) -> dict:
        schema = self.model_json_schema()
        return {
            field: field_info.get("description", "")
            for field, field_info in schema["properties"].items()
        }

    def to_yaml(self):
        # Get the model's JSON schema
        return json.loads(self.model_dump_json())

    def to_yaml_file(self, file_path):
        # Convert to YAML
        output = self.to_yaml()
        descriptions = self.get_descriptions()

        # Write to file with descriptions as a block comment at the top
        with open(file_path, "w") as file:
            for key, value in output.items():
                if key in descriptions:
                    file.write(f"# {key}: {descriptions[key]}\n")

            # then write out full object
            yaml.dump(output, file, sort_keys=False)

    @classmethod
    def from_yaml(cls, yaml_str):
        data = yaml.safe_load(yaml_str)
        return cls(**data)

    @classmethod
    def from_yaml_file(cls, file_path):
        with open(file_path, "r") as file:
            return cls.from_yaml(file.read())


class SplitBase(ModelBase):
    """
    Base class for splitting the data (i.e. Random, Dataset, Scaffold, etc)
    """

    name: str = "SplitBase"
    type_: str = "SplitBase"
    n_splits: int = Field(1, description="number of splits to generate")
    n_per_split: int = Field(..., description="Number of values per split to generate")
    deterministic: bool = Field(
        False,
        description="Whether the split is deterministic, i.e. if True it should not be run in the bootstrapping loop.",
    )
    split_level: int = Field(
        0,
        description="Level of the split, 0 indexed. The first level is applied first, and so on.",
    )

    @abc.abstractmethod
    def run(self, df: pd.DataFrame) -> [pd.DataFrame]:
        pass

    @property
    def plot_name(self) -> str:
        return f"{self.name}_{self.n_per_split}"

    def get_records(self) -> dict:
        if self.split_level == 0:
            return self._get_records()
        else:
            return {
                f"{k}_{self.split_level}": v for k, v in self._get_records().items()
            }

    @abc.abstractmethod
    def _get_records(self) -> dict:
        pass


class ReferenceStructureSplitBase(SplitBase):
    """
    Base class for splitting the data based on some attributes of the reference structure
    """

    reference_structure_column: str = Field(
        ..., description="Name of the column to distinguish reference structures by"
    )

    @abc.abstractmethod
    def run(self, df: pd.DataFrame) -> [pd.DataFrame]:
        pass

    def _get_records(self) -> dict:
        return {
            "Split": self.name,
            "N_Per_Split": self.n_per_split,
            "Reference_Structure_Column": self.reference_structure_column,
        }


class DateSplit(ReferenceStructureSplitBase):
    """
    Splits the data by date.
    """

    name: str = "DateSplit"
    type_: str = "DateSplit"
    date_dict: dict = Field(
        ...,
        description="Dictionary of dates to split the data by of the form dict[str, str] where the key is the structure name and the value is the date",
    )
    balanced: bool = Field(
        True,
        description="Whether to split the data uniformly in time (i.e. 1 split every N months) or balanced such that each split has the same number of structures",
    )
    initial_structure_error: int = Field(
        1,
        description="Initial error in the structure date. Error of 1 means no error (pick the first structure every time). Error of 20 means when you are picking up to the first 20 structures, randomize them.",
    )
    randomize_by_n_days: int = Field(
        0,
        description="Randomize the structures by n days. If 0 no randomization is done. If 1 or greater, for each structure, it can be randomly replaced by any other structure collected on that day or n-1 days from it's collection date.",
    )

    def run(self, df: pd.DataFrame) -> [pd.DataFrame]:
        # sort the structures by date
        dates = np.array(list(self.date_dict.values()))
        structures = np.array(list(self.date_dict.keys()))
        sort_idx = np.argsort(dates)
        structure_list = structures[sort_idx]
        variable_splits = []
        dfs = []
        for i in range(self.n_splits):
            start = i * self.n_per_split
            end = i * self.n_per_split + self.n_per_split

            if self.n_per_split < self.initial_structure_error:
                variable_split = np.random.choice(
                    structure_list[start : self.initial_structure_error],
                    self.n_per_split,
                    replace=False,
                )

            elif self.randomize_by_n_days > 0:
                unique_structures = df[self.reference_structure_column].unique()
                variable_split = get_unique_structures_randomized_by_date(
                    unique_structures,
                    self.date_dict,
                    self.n_per_split,
                    self.randomize_by_n_days,
                )
            else:
                variable_split = structure_list[start:end]
            variable_splits.append(variable_split)
            dfs.append(df[df[self.reference_structure_column].isin(variable_split)])
        return dfs

In [ ]:
reload(ds)

In [ ]:
def get_unique_structures_randomized_by_date(
    df: pd.DataFrame,
    structure_column: str,
    date_column: str,
    n_structures_to_return: int,
    n_days_to_randomize: int,
    date_format="%Y-%m-%d %H:%M:%S"
) -> set:
    """
    Get a set of structures randomized by date from a dataframe.

    Args:
        df: DataFrame containing structure and date information
        structure_column: Name of the column containing structure identifiers
        date_column: Name of the column containing dates
        n_structures_to_return: Number of structures to return
        n_days_to_randomize: Number of days to randomize the selection
        date_format: Format of the dates in date_column

    Returns:
        Set of selected structure identifiers
    """
    # Get unique structures
    unique_structures = df[structure_column].unique()

    if len(unique_structures) < n_structures_to_return:
        warnings.warn(
            f"Number of Unique Structures ({len(unique_structures)}) < N Structures to Return ({n_structures_to_return})."
            f"Returning all unique structures."
        )
        return set(unique_structures)

    # Create working dataframe with unique structures and their dates
    working_df = df[[structure_column, date_column]].drop_duplicates()
    working_df['date'] = pd.to_datetime(working_df[date_column], format=date_format)
    working_df.sort_values(by='date', inplace=True)

    # Get the date of the nth structure
    last_date = working_df.iloc[n_structures_to_return - 1]['date']
    last_date_with_buffer = last_date + pd.Timedelta(days=n_days_to_randomize)

    # Get all structures within the date range
    candidates = working_df[working_df['date'] <= last_date_with_buffer][structure_column].tolist()

    # Get a random sample of the candidates
    if len(candidates) > n_structures_to_return:
        candidates = np.random.choice(candidates, size=n_structures_to_return, replace=False)

    if len(candidates) < n_structures_to_return:
        raise RuntimeError(
            f"{len(candidates)} candidates < {n_structures_to_return} structures to return."
        )

    return set(candidates)

In [ ]:
docking_data

In [ ]:
unique_structures = get_unique_structures_randomized_by_date(docking_data.reference_data.dataframe,
                                            structure_column="Reference_Structure",
                                            date_column="Date",
                                            n_structures_to_return=5,
                                            n_days_to_randomize=2)

In [ ]:
cf = ds.ColumnFilter(data_type=ds.DataFrameType.REFERENCE,
                    column=ds.KeyColumn(name="Reference_Structure"),
                    value=unique_structures,
                    operator=ds.Operator.IN)

In [ ]:
docking_data.apply_filters(cf)

In [ ]:
from data_schema import ColumnFilter, Operator
def generate_random_samples(values: list, n_values: int, n_samples: int) -> list[np.ndarray]:
    """
    Generate multiple random samples from a list of values.

    Args:
        values: List of values to sample from
        n_values: Number of values to sample each time
        n_samples: Number of samples to generate

    Returns:
        List of arrays containing the sampled values
    """
    return [
        np.random.choice(values, size=n_values, replace=False)
        for _ in range(n_samples)
    ]
class RandomSplit(ReferenceStructureSplitBase):
    """
    Randomly split the structures into n_splits
    """

    name: str = "RandomSplit"
    type_: str = "RandomSplit"

    def run(self, data: ds.DockingDataModel, bootstraps=100) -> [ds.DockingDataModel]:
        docking_model = data.copy()
        ref_list = docking_model.get_unique_refs()
        random_ref_samples = generate_random_samples(ref_list, n_values=self.n_per_split, n_samples=bootstraps)
        filters = [ColumnFilter(data_type=ds.DataFrameType.REFERENCE,
                                column=ds.KeyColumn(name=self.reference_structure_column), 
                                value=sample, 
                                operator=Operator.IN) for sample in random_ref_samples]
        
        return [docking_model.apply_filters(filters=[cf]) for cf in filters]
        
class DateSplit(ReferenceStructureSplitBase):
    """
    Splits the data by date.
    """

    name: str = "DateSplit"
    type_: str = "DateSplit"
    date_column: str = Field(
        ...,
        description="Dictionary of dates to split the data by of the form dict[str, str] where the key is the structure name and the value is the date",
    )
    randomize_by_n_days: int = Field(
        0,
        description="Randomize the structures by n days. If 0 no randomization is done. If 1 or greater, for each structure, it can be randomly replaced by any other structure collected on that day or n-1 days from it's collection date.",
    )

    def run(self, data: ds.DockingDataModel) -> [pd.DataFrame]:
        # sort the structures by date
        data = data.copy()
        refdf = data.reference_data.dataframe
        date_dict = { for x in refdf[self.reference_structure_column]: x: refdf.loc[refdf[self.reference_structure_column] == x, self.date_column].values[0]}
        sort_idx = np.argsort(dates)
        structure_list = structures[sort_idx]
        variable_splits = []
        dfs = []
        if self.randomize_by_n_days > 0:
            unique_structures = df[self.reference_structure_column].unique()
            variable_split = get_unique_structures_randomized_by_date(
                unique_structures,
                self.date_dict,
                self.n_per_split,
                self.randomize_by_n_days,
            )
        else:
            variable_split = structure_list[start:end]
            variable_splits.append(variable_split)
            dfs.append(df[df[self.reference_structure_column].isin(variable_split)])
        return dfs

In [ ]:
splits = [RandomSplit(n_per_split=i, reference_structure_column="Reference_Structure")
          for i in [1,2]]

In [ ]:
new_models = [rs.run(bootstraps=1000, data=docking_data) for rs in splits]

In [ ]:
unique_refs = [model.get_unique_refs() for model in new_models]

In [ ]:
new_dfs = [docking_data.get_combined_dataframe() for docking_data in new_models]

# TODO
need to add this testing i've been doing to the test_data_model file